# Error analysis

Organized by the anticipated failure modes in protocol §7, which were written before any results existed. Each section states the prediction, then the evidence for or against it. Findings that were not predicted are kept in section 9 and are post-hoc.

Inputs are the committed outputs of `pipeline/06_run_eval.py` and `pipeline/08_score_judgments.py`. The joins are imported from 08 rather than rebuilt, so this notebook and `results/` can't disagree.

**Licensing:** section 8 can display dataset images. They are off by default (`show_images = False`). If they are turned on, clear all outputs before committing (`jupyter nbconvert --clear-output --inplace notebooks/02_error_analysis.ipynb`), since the images are not Fashionpedia's to redistribute.

In [ ]:
import importlib
import pathlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

cwd = pathlib.Path.cwd()
repo_root = next(p for p in [cwd, *cwd.parents] if (p / 'docs' / 'eval_protocol.md').exists())

# the pipeline scripts start with digits, so they're imported by name from pipeline/
sys.path.insert(0, str(repo_root / 'pipeline'))
score_judgments = importlib.import_module('08_score_judgments')

results_dir = repo_root / 'results'
image_dir = repo_root / 'data' / 'raw' / 'train'
show_images = False

precision_df = pd.read_csv(results_dir / 'precision.csv')
metrics_df = pd.read_csv(results_dir / 'metrics.csv')
crosstab_df = pd.read_csv(results_dir / 'judgment_crosstab.csv')

In [ ]:
judgments_df = pd.read_csv(score_judgments.judgments_path, dtype={'score': 'Int64', 'reason': 'string'})

# every model and condition's top 10, joined to the judgments and the reason codes
judged_df = score_judgments.add_reason_codes(score_judgments.load_judged_rankings(judgments_df))
assert len(judged_df) == 540 and judged_df['reason_code'].notna().all(), 'judged rankings incomplete'
unfiltered_df = judged_df[judged_df['condition'] == 'unfiltered']

# unique judged pairs: coded_df covers all 20 queries, pairs_df adds metadata relevance for Tier A/B/C
coded_df = score_judgments.add_reason_codes(judgments_df)
pairs_df = score_judgments.add_reason_codes(score_judgments.label_pairs(judgments_df))

print(f"judged rankings: n = {len(judged_df)}, unique pairs: n = {len(coded_df)}, Tier A/B/C pairs: n = {len(pairs_df)}")

In [ ]:
def by_model(df, values, condition='unfiltered'):
    # one row per query, one column per model
    return df[df['condition'] == condition].pivot(index='query_id', columns='model_key', values=values)

def code_counts(df, query_ids, scores=(0, 1)):
    # reason codes per query and score, counted separately for each model's top 10
    subset = df[df['query_id'].isin(query_ids) & df['score'].isin(list(scores))]
    return subset.groupby(['query_id', 'score', 'reason_code', 'model_key']).size().unstack('model_key', fill_value=0)

## 1. Tier D and the v1 scope decision (§3.3, §6)

**Prediction:** Tier D is "EXPECTED TO FAIL" and is the evidence base for leaving style and genre classification out of v1. If it performs well, the scope decision is re-examined in the write-up rather than defended.

Tier D mixes three kinds of query: color (D1), occasion (D2, D3) and style/genre (D4). Only D4 is the style classification that v1 leaves out of scope, so the queries are read separately, not as one average. Tier D has no metadata ground truth, so manual precision is the only evidence.

In [ ]:
by_model(precision_df[precision_df['tier'] == 'D'], ['strict_precision@10', 'lenient_precision@10']).round(2)

In [ ]:
code_counts(unfiltered_df, ['D1', 'D2', 'D3', 'D4'], scores=[0, 1, 2])

**Observed.** The prediction holds for style and fails for color and occasion. FashionCLIP's strict precision@10 is 1.0 on D1 (color) and D2 (occasion) and 0.9 on D3, against 0.5 on D4 (`goth`), where both models land. CLIP is lower on D1 to D3 (0.8, 0.7, 0.3), and half of its D3 top 10 is `summery, not beach`. The scope decision therefore stands, but on narrower grounds than §6 stated: the evidence is D4 alone, not Tier D as a whole. That is four queries of ten images each, judged by one person, and D2 is subjective by the protocol's own account.

## 2. Small garments are diluted by whole-image embedding (§7.1)

**Prediction:** queries for small or peripheral items suffer most: A4 `scarf`, A5 `umbrella`, and the accessory half of C1 (boots) and C3 (scarf).

For the accessory half, the partial-match codes say which conjunct was missing. In C1, `skinny jeans, no boots` means the accessory half was missed. In C3, `long coat, no scarf` means the accessory half was missed, and `scarf, coat too short` means the garment half was missed.

In [ ]:
by_model(metrics_df[metrics_df['tier'] == 'A'], 'capped_recall@10')

In [ ]:
code_counts(unfiltered_df, ['C1', 'C3'], scores=[1])

**Observed.** Mixed. For single-item queries the prediction does not hold: A4 `scarf` scores 0.9 (CLIP) and 0.8 (FashionCLIP), and A5 `umbrella` scores 1.0 for both, against an A1 to A3 range of 0.5 to 1.0. For the accessory half of compositional queries it splits. In C1 the partial matches are missing the boots (`skinny jeans, no boots`: 2 CLIP, 3 FashionCLIP), which is consistent with §7.1. In C3 the scarf is usually found and the coat length is what fails (`scarf, coat too short`: 3 CLIP, 6 FashionCLIP, against `long coat, no scarf`: 1 each), which runs against it.

## 3. Label sparsity depresses metadata recall (§7.2)

**Prediction:** wherever manual precision@10 is substantially higher than capped recall@10 on the same query, label sparsity is the leading explanation. The gap is expected to be widest on B3 and B6.

§7.2 does not say strict or lenient, so both gaps are in `results/precision.csv`. The ranking below uses the strict gap, averaged over the two models.

In [ ]:
gap_df = precision_df[(precision_df['condition'] == 'unfiltered') & (precision_df['tier'] != 'D')]

gap_wide = gap_df.pivot(index='query_id', columns='model_key', values='strict_gap')
gap_wide['mean_strict_gap'] = gap_wide.mean(axis=1)
gap_wide = gap_wide.sort_values('mean_strict_gap', ascending=False)

predicted = {'B3', 'B6'}
observed = set(gap_wide['mean_strict_gap'].nlargest(2).index)
print(f"predicted widest: {sorted(predicted)}, observed widest: {sorted(observed)}")

gap_wide.round(2)

In [ ]:
ax = gap_df.pivot(index='query_id', columns='model_key', values='strict_gap').plot.bar(figsize=(10, 4))
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('query')
ax.set_ylabel('strict precision@10 minus capped recall@10')
ax.set_title('Label sparsity gap by query, unfiltered')
plt.tight_layout()
plt.show()

In [ ]:
# metadata relevance against manual score, unique pairs, for the predicted queries and the observed widest
crosstab_df[crosstab_df['query_id'].isin(sorted(predicted | observed))]

**Observed.** The prediction is half right. B3 ties with C1 for the second-widest gap (0.2 CLIP, 0.3 FashionCLIP). B6 has none (−0.1 and 0.0): of its judged pairs, the predicate missed only one, and two pairs it counted as relevant were judged `clean denim`. The widest gap, on B1 (0.9 and 1.0), was not predicted: 18 of the 19 B1 pairs outside the predicate were judged clearly relevant, which matches the amendment note that sleeveless dresses often have no sleeve instance to label. B2 has no gap for either model. On every B2 pair in either model's unfiltered top 10, the predicate and the hand score agree, because the predicate's `shirt, blouse` category correctly excludes the striped t-shirts that fill CLIP's top 10 (section 9.4).

## 4. Confusion between visually similar categories (§7.3)

**Prediction:** `cardigan` vs `sweater` vs `jacket` (A1), `shorts` vs `skirt` (A3), `jacket` vs `coat` (C4 vs C3), and `pants` vs `tights, stockings` (B3).

The `wrong garment:` codes record what a non-matching image showed instead. A query with no score-0 or score-1 images has no confusion to report.

In [ ]:
coded_df[coded_df['query_id'].isin(['A1', 'A3', 'B3', 'C3', 'C4'])].groupby('query_id')['score'].value_counts().unstack(fill_value=0)

In [ ]:
code_counts(unfiltered_df, ['A1', 'A3', 'B3', 'C3', 'C4'])

**Observed.**
- `cardigan` vs `sweater` vs `jacket` (A1): confirmed for CLIP only, whose four score-0 images were a sweater, a jacket, a blazer and a coat. FashionCLIP's A1 top 10 has none.
- `shorts` vs `skirt` (A3): not observed. Every A3 pair scored 2.
- `pants` vs `tights, stockings` (B3): not observed. B3's errors are waistline errors (`mid rise`, `low rise`, `not high waisted`), not category errors; the one category error is a jumpsuit.
- `jacket` vs `coat` (C4 vs C3): observed on inspection of the images, though the codes alone do not show it. C4's `no blazer` code does not record what the garment was. In the images, three of FashionCLIP's four `no blazer` results are long coats, and CLIP's four are two cardigans, a long coat and a trench coat. C3's `scarf, not coat` was never used.

## 5. Scene and context override garment (§7.4)

**Prediction:** queries with strong scene associations, D3 especially, retrieve images that match the context while missing the garment.

D3's prose counts a clear beach setting as relevant on its own, so a scene-only match still scores 2. Judgments that qualified *only* through the setting were coded `beach setting only`, separate from the garment codes.

In [ ]:
n_scene_only = (coded_df['reason_code'] == 'beach setting only').sum()
print(f"D3 judgments that qualified on setting alone: n = {n_scene_only}")

code_counts(unfiltered_df, ['D3'], scores=[0, 1, 2])

**Observed.** Not observed. No D3 judgment qualified on setting alone. CLIP's D3 misses are garment-level (`summery, not beach`: 5 of 10), not scene matches without beachwear.

## 6. Compositional queries collapse to one term (§7.5)

**Prediction:** for C1 through C3, retrieval is dominated by whichever term is more salient, returning images that match one conjunct only. Diagnosed through the score-1 (partial match) rate.

In [ ]:
c_df = unfiltered_df[unfiltered_df['query_id'].isin(['C1', 'C2', 'C3'])]
c_df.groupby(['query_id', 'model_key'])['score'].value_counts(normalize=True).unstack(fill_value=0).round(2)

In [ ]:
code_counts(unfiltered_df, ['C1', 'C2', 'C3'], scores=[1])

**Observed.** Consistent with the prediction. Partial matches make up 0.3 of C1's top 10 for both models, 0.5 of C2's, and 0.5 (CLIP) and 0.7 (FashionCLIP) of C3's. The dominant term differs by query: the jeans in C1 (boots missing), the blazer in C2 (`blazer, not t-shirt`: 5 CLIP, 2 FashionCLIP), and the scarf in C3 (coat too short).

## 7. Corpus coverage gaps (§7.6)

**Prediction:** some queries have few true instances in Fashionpedia regardless of model quality. `n_relevant` separates a low score caused by an empty corpus from one caused by bad retrieval. Where `n_relevant` is below 10, capped recall@10 equals plain recall and the ceiling is still 1.0.

In [ ]:
coverage = by_model(metrics_df, 'capped_recall@10')
coverage.insert(0, 'n_relevant', metrics_df.groupby('query_id')['n_relevant'].first())
coverage.sort_values('n_relevant')

**Observed.** C3 is the one query where coverage is a plausible factor: only 16 images in the corpus satisfy its predicate, against at least 112 for every other query. B1 is not a coverage case. It has 112 relevant images by metadata and 0.9 to 1.0 strict precision, so its zero recall is label sparsity (section 3), not an empty corpus.

## 8. Failure cases (§4)

Any query with capped recall@10 below 0.30 is a failure case, with its retrieved images inspected and its failure mode categorized against §7. The table pulls the evidence from sections 2 to 7 together. The image grid below is off by default; see the licensing note at the top.

In [ ]:
failure_threshold = 0.30
failures = metrics_df[(metrics_df['condition'] == 'unfiltered') & (metrics_df['capped_recall@10'] < failure_threshold)]

failure_df = failures[['model_key', 'query_id', 'n_relevant', 'capped_recall@10', 'first_relevant_rank']].merge(
    precision_df[precision_df['condition'] == 'unfiltered'][['model_key', 'query_id', 'strict_precision@10', 'lenient_precision@10']],
    on=['model_key', 'query_id'], how='left', validate='1:1')

failure_df.sort_values(['query_id', 'model_key'])

**Categorization.** Based on the metrics, the reason codes, and inspection of each failure case's top 10 images (`show_images = True`).

| Query | Model | Leading mode | Evidence |
|---|---|---|---|
| B1 | both | §7.2 label sparsity | recall 0.0, strict precision 0.9 and 1.0 |
| B2 | CLIP | retrieval failure, shirt vs t-shirt (section 9.4) | recall 0.1, strict precision 0.1; 7 `striped tee, not shirt` |
| B3 | FashionCLIP | §7.2 sparsity plus real waistline errors | recall 0.2, strict 0.5; three `mid rise` or `not high waisted`, one jumpsuit |
| B7 | CLIP | retrieval failure, fit term ignored | strict 0.1, `straight leg` 5, `slim fit` 2 |
| C3 | both | §7.5 compositional, with §7.6 coverage | 16 relevant images; scarf found, coat too short |
| C4 | both | retrieval failure, fit term ignored | strict 0.0; `no blazer` 4 each, `fitted blazer` 5 for CLIP |

B7 and C4 share a pattern §7 did not name: the fit modifier (`baggy`, `oversized`) is dropped and the garment term alone is matched. That is post-hoc and is reported as such in section 9.

In [ ]:
code_counts(unfiltered_df[unfiltered_df['model_key'].isin(failures['model_key'])], sorted(failures['query_id'].unique()))

In [ ]:
def show_top_10(model_key, query_id):
    top_df = unfiltered_df[(unfiltered_df['model_key'] == model_key) & (unfiltered_df['query_id'] == query_id)].sort_values('rank')
    top_df = top_df.merge(judgments_df[['query_id', 'image_id', 'file_name']], on=['query_id', 'image_id'], validate='1:1')

    fig, axes = plt.subplots(2, 5, figsize=(15, 7))
    for ax, row in zip(axes.flat, top_df.itertuples()):
        ax.imshow(Image.open(image_dir / row.file_name))
        ax.set_title(f"#{row.rank}: score {row.score}\n{row.reason_code}", fontsize=9)
        ax.axis('off')

    fig.suptitle(f"{model_key}, {query_id}, unfiltered top 10")
    plt.tight_layout()
    plt.show()

if show_images:
    for model_key, query_id in failures[['model_key', 'query_id']].itertuples(index=False):
        show_top_10(model_key, query_id)

## 9. Post-hoc findings

Not predicted in §7. Reported separately, as §4 and §8 require.

### 9.1 Metadata says relevant, judged not relevant

Pairs the predicate counts as relevant but that scored 0 by hand. These are label noise or predicates broader than the prose, and they are what produce the negative gaps in section 3.

In [ ]:
pairs_df[pairs_df['is_relevant'] & (pairs_df['score'] == 0)][['query_id', 'judge_order', 'image_id', 'reason_code']].sort_values('query_id')

**Observed.** Six pairs. Two are B6 images judged `clean denim`. B6's predicate accepts `298:washed` alongside `distressed` and `frayed`, so a washed but undamaged pair of jeans can satisfy the predicate without meeting the prose; that is a plausible cause, to confirm on the images. The other four (A2 separates labeled as a jumpsuit, B4 vest, B5 butterfly print labeled floral, B7 non-denim labeled jeans) look like annotation noise.

### 9.2 Metadata says not relevant, judged clearly relevant

Pairs the predicate misses but that scored 2 by hand, per query. This is the label-sparsity evidence behind section 3, at the level of unique pairs.

In [ ]:
missed = pairs_df[~pairs_df['is_relevant']]
missed_by_query = missed.groupby('query_id')['score'].value_counts().unstack(fill_value=0).reindex(columns=[0, 1, 2], fill_value=0)
missed_by_query['share_scored_2'] = (missed_by_query[2] / missed_by_query.sum(axis=1)).round(2)
missed_by_query.sort_values('share_scored_2', ascending=False)

**Observed.** Predicate misses judged clearly relevant are concentrated in B1 (18 of 19), with smaller shares in B3 (6 of 16), C1 (5 of 12) and B4 (3 of 8). For Tier A the misses are real (no score-2 pairs among them), so Tier A's metadata is effectively complete for these queries.

### 9.3 Fit modifiers are dropped

Not named in §7. On B7 (`baggy jeans`) and C4 (`oversized blazer`), the misses are mostly the right garment in the wrong fit: `straight leg` and `slim fit` on B7, `fitted blazer` and `slightly large` on C4. FashionCLIP handles B7 better (strict 0.7 against CLIP's 0.1); on C4 neither model returns a clearly oversized blazer. The B7/C1 pairing in §6 was designed to detect exactly this.

### 9.4 Shirt vs t-shirt (B2)

Not named in §7. CLIP's unfiltered B2 top 10 holds seven striped t-shirts and one striped knit top, so its strict precision (0.1) matches its capped recall (0.1). FashionCLIP's top 10 holds nine striped shirts or blouses. Eight of these B2 judgments were corrected from 2 to 1 after inspection of the images, because they broke the t-shirt rule set during B2 judging; before the correction, B2 looked like a label-sparsity case for CLIP. The correction is recorded in the git history of `docs/relevance_judgments.csv`.